In [1]:
!python --version

Python 3.11.4


In [2]:
print("Hello World!")

Hello World!


In [3]:
5+9*34

311

In [2]:
import findspark
findspark.init()
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

# Create SparkSession with Hudi configuration

spark = SparkSession.builder \
    .appName("HudiExample") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/09/22 12:08:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import Row
from pyspark.sql.functions import lit
# Example data
data = [Row(id=1, name="Alice", age=24), Row(id=2, name="Bob", age=30)]
df = spark.createDataFrame(data)



In [20]:
dfWithTimestamp = df.withColumn("curr_timestamp", f.current_timestamp())
dfWithTimestamp.show()

+---+-----+---+--------------------+
| id| name|age|      curr_timestamp|
+---+-----+---+--------------------+
|  1|Alice| 24|2024-09-21 15:51:...|
|  2|  Bob| 30|2024-09-21 15:51:...|
+---+-----+---+--------------------+



In [43]:
# Write data to Hudi table
hudi_options = {
    'hoodie.table.name': 'my_hudi_table',
    'hoodie.datasource.write.recordkey.field': 'id',
    'hoodie.datasource.write.partitionpath.field': 'curr_timestamp',
    'hoodie.datasource.write.table.type': 'COPY_ON_WRITE',
    'hoodie.datasource.write.precombine.field': 'curr_timestamp',
    'hoodie.datasource.hive_sync.enable': 'false',  # Disable Hive Sync
    'hoodie.metadata.enable': 'true'
}
dfWithTimestamp.write.format("hudi").options(**hudi_options).mode("overwrite").save("/home/sparkuser/app/hudi/table")


24/09/21 22:26:41 WARN HoodieSparkSqlWriterInternal: hoodie table at /home/sparkuser/app/hudi/table already exists. Deleting existing data & overwriting with new data.
24/09/21 22:26:45 WARN HoodieSparkSqlWriterInternal: Closing write client


In [50]:
# Read Hudi data back
hudi_read_options = {
    'hoodie.datasource.query.type': 'snapshot',
    'hoodie.metadata.enable': 'true'
}

df_hudi = spark.read.format("hudi").options(**hudi_read_options).load("/home/sparkuser/app/hudi/table/*")
df_hudi.show(truncate=True)

+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+
|_hoodie_commit_time|_hoodie_commit_seqno|_hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name| id|   name|age|      curr_timestamp|
+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+
|  20240921222641401|20240921222641401...|                 2|      1726957601430937|5901e2d7-743d-4a4...|  2|    Bob| 30|2024-09-21 22:26:...|
|  20240921222641401|20240921222641401...|                 1|      1726957601430937|5901e2d7-743d-4a4...|  1|  Alice| 24|2024-09-21 22:26:...|
|  20240921222655270|20240921222655270...|                 3|      1726957615334409|d3bbf95c-32e2-496...|  3|Charlie| 28|2024-09-21 22:26:...|
+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+

In [51]:
# Adding a new record
new_data = [Row(id=3, name="Charlie", age=28)]
df_new = spark.createDataFrame(new_data).withColumn("curr_timestamp", f.current_timestamp())
df_new.show()

# Write data (upsert)
df_new.write.format("hudi").options(**hudi_options).mode("append").save("/home/sparkuser/app/hudi/table")


+---+-------+---+--------------------+
| id|   name|age|      curr_timestamp|
+---+-------+---+--------------------+
|  3|Charlie| 28|2024-09-21 22:28:...|
+---+-------+---+--------------------+



24/09/21 22:28:55 WARN HoodieTableFileSystemView: Partition: 1726957734174439 is not available in store
24/09/21 22:28:55 WARN HoodieTableFileSystemView: Partition: 1726957734174439 is not available in store
24/09/21 22:28:57 WARN HoodieSparkSqlWriterInternal: Closing write client


In [46]:
df_hudi = spark.read.format("hudi").options(**hudi_read_options).load("/home/sparkuser/app/hudi/table/*")
df_hudi.show()

+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+
|_hoodie_commit_time|_hoodie_commit_seqno|_hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name| id|   name|age|      curr_timestamp|
+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+
|  20240921222641401|20240921222641401...|                 2|      1726957601430937|5901e2d7-743d-4a4...|  2|    Bob| 30|2024-09-21 22:26:...|
|  20240921222641401|20240921222641401...|                 1|      1726957601430937|5901e2d7-743d-4a4...|  1|  Alice| 24|2024-09-21 22:26:...|
|  20240921222655270|20240921222655270...|                 3|      1726957615334409|d3bbf95c-32e2-496...|  3|Charlie| 28|2024-09-21 22:26:...|
+-------------------+--------------------+------------------+----------------------+--------------------+---+-------+---+--------------------+

In [1]:
# Load Hudi metadata table using the `hudi_metadata` format
# metadata_df = spark.read.format("hudi_metadata").load("/home/sparkuser/app/hudi/table/.hoodie/metadata")
# metadata_df.show(truncate=False)

spark.sql("SELECT table_name, partition_path, file_count FROM information_schema.hudi_table_metadata WHERE table_name = 'my_hudi_table'").show()

NameError: name 'spark' is not defined

In [4]:
# Query to view partitions in the Hudi table
# metadata_df.createOrReplaceTempView("metadata_table") 
spark.sql("SELECT * FROM information_schema.hudi_table_metadata").show(truncate=False)


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `information_schema`.`hudi_table_metadata` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [information_schema, hudi_table_metadata], [], false
